In [4]:
# This is the preprocessing script. It basiclaly takes the original 2 files (S&P500 open/close, historic) 
# and federal speeches, transforms their date to be mergeable, and then constructs our DV and appends on the file
# Then this script also processes the speeches, as they are webscraped by origin and put on kaggle (we did not scrape)
# So we remove references, videolinks and other items that we simply CANNOT let enter our NBC, LDA and Word2Vec models.

# First, pandas for dataframes.
import pandas as pd


# We then read our first csv, historic s&p500 opens and closes ranging back nearly 100 years.
sp = pd.read_csv("s_p500_historic.csv")

# The dates look like "12/30/27", with a 2-digit year. The data runs from 1927
# to 2020, so a year of 20 or less means 20xx and anything above 20 means 19xx.
# The data runs from 1927 to 2020, but has 2-digit years. we need to construct a 4-digit to not confuse 27 (1927) with (2027) 
# for the model and also sorting (data leakage, more on this later)


# FIrst we create an empty list to store real dates that are the output of the for loop
real_dates = []

for date_text in sp["Date"]: # for each "date_text" in the s&p date column
    month, day, year = date_text.split("/") # break it into pieces
    year = int(year) # and convert year into an actual number from text
    if year <= 20: # if year is 20 or smaller,
        year = 2000 + year # assume it is 2000s
    else:
        year = 1900 + year # else, its 1900's
    real_dates.append(pd.Timestamp(year=year, month=int(month), day=int(day))) # now first we create pandas timestamp
                                        # and then we convert the months and days into numbers(integers) and append it
                                        # to our empty list we created above
    
# and now overwrite the date with the newly transformed date
sp["Date"] = real_dates

# For many reasons, we sort the speeches now by date to e.g. train on early, test on later data.
sp = sp.sort_values("Date")


# Now we move into the fed speeches.
fed = pd.read_csv("fed_speeches_1996_2020.csv")


# First, we drop every row that has an empty cell in any column.
fed = fed.dropna()

# The dates are numbers like 19961219, so we turn them into real dates
fed["speech_date"] = pd.to_datetime(fed["date"], format="%Y%m%d")


# Now, both our dataframes are matching in date, so we can construct our DV.

# First, we create the two lists that can store the closing prices form the loop below
close_before = []
close_after = []


# Then we create the loop
for speech_date in fed["speech_date"]:
    earlier_days = sp[sp["Date"] < speech_date]    # finding a list of all trading days before speech
    later_days = sp[sp["Date"] > speech_date]      # finding a list of all trading days after speech
    close_before.append(earlier_days.iloc[-1]["Close"])   # then appends the LAST day before speech (the close)
    close_after.append(later_days.iloc[0]["Close"])       # and appends the FIRST trading day after speech (close)
    
# now attach the lists we made to columns
fed["close_before"] = close_before
fed["close_after"] = close_after
# creating our DV
fed["S&PREACT"] = (fed["close_after"] - fed["close_before"]) / fed["close_before"] * 100


# After manual inspection of speeches, we found issues related to unwanted text as a cause from the dataset being scraped.
# Now we get to the part where we clean the speeches
# The words below mark where the speeches end (often)
# And since most speeches are followed by references footnotes and website footer, we need to remove these
# in order not to skew our results with our 3 models.

speech_ends_at = ["References", "REFERENCES", "Footnotes", "Return to text",
                  "View speech charts and figures", "Accessible Version",
                  "Home | News", "Last update:"]

# again we create empty list that will be used to store the output of for loop below
clean_texts = []

# for each speech, we want to do these steps and clean them
for text in fed["text"]:

  # this next part determines if characters are normal of english alphabet or broken, by length of their code. 
    # under 128 means normal
    normal_text = ""
    for character in text:
        if ord(character) < 128:
            normal_text += character # if character has code under 128, add it to normal_text, 
        else:
            normal_text += " " # and if it does not, add a space in its place instead
    text = " ".join(normal_text.split()) # Collapses all irregular spaces down to a single space between words

    # Some speeches start with the code of the websites video player
    # The code always ends the same way, so we keep only what comes after it.
    if "myPlayer.play(); } }" in text:
        text = text.split("myPlayer.play(); } }")[1]

    # This loop chops off everything after end of speech, as we defined
    for marker in speech_ends_at:
        text = text.split(marker)[0] 

    clean_texts.append(text) # This one then adds the final cleaned speech t the clean_texts list we created above

fed["text"] = clean_texts # And now we can overwrite the column to the cleaned speeches


# Some speeches are super short (or not speeches) such as a website link that failed to be scraped
# We remove these by ensuring that length is atleast 1000 characters. Any speech is atleast 1000 characters.
fed = fed[fed["text"].str.len() >= 1000]

# Some speeches appear twice, so here we remove them.
fed = fed.drop_duplicates(subset="text")


# At last, we just only keep columns that we will want to use, to make our models more efficient and cut down size of final file
fed = fed[["text", "speech_date", "speaker", "S&PREACT"]]

# And we save the final file so each model can load this and terraform (if necessary)
# Also why this one does not clean stopwords, as some will use stopwords (word2vec) and some will not (Lda)
fed.to_csv("fed_speeches_clean.csv", index=False)

# Size of the final dataset (rows, columns)
print("Shape of dataset:", fed.shape)

# Mean and standard deviation of our DV
print("Mean of S&PREACT :", fed["S&PREACT"].mean())
print("Standard deviation of DV :", fed["S&PREACT"].std())

Shape of dataset: (1433, 4)
Mean of S&PREACT : 0.07062007659751193
Standard deviation of DV : 1.5626482776487765
